<a href="https://colab.research.google.com/github/aryaveermajumdar/fsfm-based-neutral-face-project/blob/main/FSFM_Demographic_Experiments_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ============================================================
# MASTER SETUP — Demographic Pipeline Notebook
# ============================================================
import os, sys, torch
import torchvision.transforms as T
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from collections import Counter
from google.colab import drive

drive.mount('/content/drive')

# GPU check
print("CUDA:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0))

# Paths
DRIVE_BASE = '/content/drive/MyDrive/Colab Notebooks'
DEMO_CACHE = f'{DRIVE_BASE}/demographic_classifier'
os.makedirs(DEMO_CACHE, exist_ok=True)

# FSFM transform
MEAN = [0.5482207536697388, 0.42340534925460815, 0.3654651641845703]
STD  = [0.2789176106452942, 0.2438540756702423,  0.23493893444538116]
transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=MEAN, std=STD)
])

print("Setup complete.")

Mounted at /content/drive
CUDA: True
GPU: Tesla T4
Setup complete.


In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from collections import Counter
import os

# --- Constants ---
RACE_NAMES_4 = {0: 'White/Middle Eastern', 1: 'Black',
                2: 'Asian', 3: 'Indian/Latino'}
AGE_NAMES_3  = {0: 'Young', 1: 'Middle', 2: 'Old'}
GENDER_NAMES = {0: 'Male', 1: 'Female'}

AGE_TO_3CLASS = {
    0: 0, 1: 0, 2: 0,  # 0-2, 3-9, 10-19 → Young
    3: 1, 4: 1, 5: 1,  # 20-29, 30-39, 40-49 → Middle
    6: 2, 7: 2, 8: 2,  # 50-59, 60-69, 70+ → Old
}

RAFDB_BASE_CACHE = '/content/drive/MyDrive/Colab Notebooks/RAF-DB'
DEMO_CACHE       = '/content/drive/MyDrive/Colab Notebooks/demographic_classifier'

# --- Helper functions ---
def remap_races(cache, mapping):
    new_cache = dict(cache)
    new_cache['races'] = torch.tensor(
        [mapping[r.item()] for r in cache['races']],
        dtype=torch.long
    )
    return new_cache

def remap_ages(cache, mapping):
    new_cache = dict(cache)
    new_cache['ages'] = torch.tensor(
        [mapping[a.item()] for a in cache['ages']],
        dtype=torch.long
    )
    return new_cache

# --- Model definition ---
class DemographicClassifier4v2(nn.Module):
    def __init__(self, feature_dim=768, hidden_dim=512,
                 num_races=4, num_genders=2, num_ages=3, dropout=0.3):
        super().__init__()
        self.trunk = nn.Sequential(
            nn.Linear(feature_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 256),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.race_head   = nn.Linear(256, num_races)
        self.gender_head = nn.Linear(256, num_genders)
        self.age_head    = nn.Linear(256, num_ages)

    def forward(self, x):
        shared = self.trunk(x)
        return (self.race_head(shared),
                self.gender_head(shared),
                self.age_head(shared))

# --- Load FairFace caches ---
train_ff_4 = torch.load(f'{DEMO_CACHE}/fairface_train_4class.pt',
                         weights_only=False)
val_ff_4   = torch.load(f'{DEMO_CACHE}/fairface_val_4class.pt',
                         weights_only=False)
train_ff_4 = remap_ages(train_ff_4, AGE_TO_3CLASS)
val_ff_4   = remap_ages(val_ff_4,   AGE_TO_3CLASS)

# --- Load demographic classifier ---
demo_clf4v2 = DemographicClassifier4v2().cuda()
demo_clf4v2.load_state_dict(
    torch.load(f'{DEMO_CACHE}/demographic_classifier_4race_3age.pt',
               weights_only=False)
)
demo_clf4v2.eval()
print("Demographic classifier loaded.")

# --- Load unified train + RAF-DB test ---
unified_train = torch.load(f'{DEMO_CACHE}/unified_train.pt',
                            weights_only=False)
test_raf      = torch.load(f'{RAFDB_BASE_CACHE}/test_features_4race_3age.pt',
                            weights_only=False)

# --- Fix gender clamp ---
unified_train['genders'] = unified_train['genders'].clamp(0, 1)
test_raf['genders']      = test_raf['genders'].clamp(0, 1)

# --- Verify ---
print("Unified train:", unified_train['features'].shape)
print("Test RAF-DB:",   test_raf['features'].shape)
print("Genders max — train:", unified_train['genders'].max().item(),
      "test:", test_raf['genders'].max().item())

Demographic classifier loaded.
Unified train: torch.Size([23226, 768])
Test RAF-DB: torch.Size([3068, 768])
Genders max — train: 1 test: 1


In [4]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import label_binarize
from collections import Counter
import numpy as np

UNIFIED_CLASSES = ['Surprise','Fear','Disgust',
                   'Happiness','Sadness','Anger','Neutral']
GENDER_MAP = {0: 'Male', 1: 'Female'}

train_loader = DataLoader(
    TensorDataset(
        unified_train['features'], unified_train['labels'],
        unified_train['races'],    unified_train['genders'],
        unified_train['ages']
    ),
    batch_size=256, shuffle=True
)
test_loader = DataLoader(
    TensorDataset(
        test_raf['features'], test_raf['labels'],
        test_raf['races'],    test_raf['genders'],
        test_raf['ages']
    ),
    batch_size=256, shuffle=False
)

counts  = Counter(unified_train['labels'].tolist())
total_n = sum(counts.values())
weights = torch.tensor(
    [total_n/counts[i] for i in range(7)], dtype=torch.float
).cuda()
weights = weights / weights.sum()

class StagedEmotionHead(nn.Module):
    def __init__(self, feature_dim=768,
                 num_races=4, num_genders=2, num_ages=3,
                 embed_dim=32, hidden_dim=512,
                 num_emotions=7, dropout=0.3):
        super().__init__()
        self.race_emb   = nn.Embedding(num_races,   embed_dim)
        self.gender_emb = nn.Embedding(num_genders, embed_dim)
        self.age_emb    = nn.Embedding(num_ages,    embed_dim)
        self.race_aux   = nn.Linear(embed_dim, num_races)
        self.gender_aux = nn.Linear(embed_dim, num_genders)
        self.age_aux    = nn.Linear(embed_dim, num_ages)
        combined_dim = feature_dim + embed_dim * 3
        self.classifier = nn.Sequential(
            nn.Linear(combined_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_emotions)
        )

    def forward(self, feats, races, genders, ages):
        combined = torch.cat([
            feats,
            self.race_emb(races),
            self.gender_emb(genders),
            self.age_emb(ages)
        ], dim=-1)
        return self.classifier(combined)

    def forward_demo(self, races, genders, ages):
        return (self.race_aux(self.race_emb(races)),
                self.gender_aux(self.gender_emb(genders)),
                self.age_aux(self.age_emb(ages)))


def set_seed(seed):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)


def train_and_evaluate(seed):
    set_seed(seed)
    model = StagedEmotionHead().cuda()
    criterion_emotion = nn.CrossEntropyLoss(weight=weights)
    criterion_demo    = nn.CrossEntropyLoss()

    # Stage 1
    for param in model.classifier.parameters():
        param.requires_grad = False
    stage1_params = (
        list(model.race_emb.parameters()) +
        list(model.gender_emb.parameters()) +
        list(model.age_emb.parameters()) +
        list(model.race_aux.parameters()) +
        list(model.gender_aux.parameters()) +
        list(model.age_aux.parameters())
    )
    opt1 = optim.AdamW(stage1_params, lr=1e-3, weight_decay=0.05)
    for _ in range(15):
        model.train()
        for feats, labels, races, genders, ages in train_loader:
            races, genders, ages = races.cuda(), genders.cuda(), ages.cuda()
            r_l, g_l, a_l = model.forward_demo(races, genders, ages)
            loss = (criterion_demo(r_l, races) +
                    criterion_demo(g_l, genders) +
                    criterion_demo(a_l, ages))
            opt1.zero_grad(); loss.backward(); opt1.step()

    # Stage 2
    for param in model.race_emb.parameters():   param.requires_grad = False
    for param in model.gender_emb.parameters(): param.requires_grad = False
    for param in model.age_emb.parameters():    param.requires_grad = False
    for param in model.race_aux.parameters():   param.requires_grad = False
    for param in model.gender_aux.parameters(): param.requires_grad = False
    for param in model.age_aux.parameters():    param.requires_grad = False
    for param in model.classifier.parameters(): param.requires_grad = True
    opt2 = optim.AdamW(model.classifier.parameters(),
                       lr=1e-3, weight_decay=0.05)
    for _ in range(40):
        model.train()
        for feats, labels, races, genders, ages in train_loader:
            feats, labels = feats.cuda(), labels.cuda()
            races, genders, ages = races.cuda(), genders.cuda(), ages.cuda()
            loss = criterion_emotion(
                model(feats, races, genders, ages), labels)
            opt2.zero_grad(); loss.backward(); opt2.step()

    # Stage 3
    for param in model.parameters():
        param.requires_grad = True
    opt3 = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.05)
    for _ in range(20):
        model.train()
        for feats, labels, races, genders, ages in train_loader:
            feats, labels = feats.cuda(), labels.cuda()
            races, genders, ages = races.cuda(), genders.cuda(), ages.cuda()
            loss = criterion_emotion(
                model(feats, races, genders, ages), labels)
            opt3.zero_grad(); loss.backward(); opt3.step()

    # Evaluate
    model.eval()
    all_preds, all_labels_e = [], []
    all_races_e, all_genders_e, all_ages_e = [], [], []
    all_probs = []

    with torch.no_grad():
        for feats, labels, races, genders, ages in test_loader:
            feats = feats.cuda()
            races, genders, ages = races.cuda(), genders.cuda(), ages.cuda()
            logits = model(feats, races, genders, ages)
            probs  = torch.softmax(logits, dim=1).cpu()
            all_probs.extend(probs.tolist())
            all_preds.extend(logits.argmax(1).cpu().tolist())
            all_labels_e.extend(labels.tolist())
            all_races_e.extend(races.cpu().tolist())
            all_genders_e.extend(genders.cpu().tolist())
            all_ages_e.extend(ages.cpu().tolist())

    all_probs    = np.array(all_probs)
    all_preds    = np.array(all_preds)
    all_labels_e = np.array(all_labels_e)
    all_races_e  = np.array(all_races_e)
    all_genders_e = np.array(all_genders_e)
    all_ages_e   = np.array(all_ages_e)

    y_bin     = label_binarize(all_labels_e, classes=list(range(7)))
    acc       = (all_preds == all_labels_e).mean()
    macro_auc = roc_auc_score(y_bin, all_probs,
                               multi_class='ovr', average='macro')

    neutral_id = 6
    neutral_recalls = {}

    # Race
    for gid, gname in RACE_NAMES_4.items():
        mask = all_races_e == gid
        if mask.sum() == 0: continue
        true_n = (all_labels_e[mask] == neutral_id)
        pred_n = (all_preds[mask]    == neutral_id)
        if true_n.sum() == 0: continue
        neutral_recalls[f'Race:{gname}'] = \
            (true_n & pred_n).sum() / true_n.sum()

    # Gender
    for gid, gname in GENDER_MAP.items():
        mask = all_genders_e == gid
        if mask.sum() == 0: continue
        true_n = (all_labels_e[mask] == neutral_id)
        pred_n = (all_preds[mask]    == neutral_id)
        if true_n.sum() == 0: continue
        neutral_recalls[f'Gender:{gname}'] = \
            (true_n & pred_n).sum() / true_n.sum()

    # Age
    for gid, gname in AGE_NAMES_3.items():
        mask = all_ages_e == gid
        if mask.sum() == 0: continue
        true_n = (all_labels_e[mask] == neutral_id)
        pred_n = (all_preds[mask]    == neutral_id)
        if true_n.sum() == 0: continue
        neutral_recalls[f'Age:{gname}'] = \
            (true_n & pred_n).sum() / true_n.sum()

    return acc, macro_auc, neutral_recalls


# --- Run 8 seeds ---
SEEDS = [42, 67, 69, 123, 420, 456, 789, 1024, 69420]
seed_results = []

for seed in SEEDS:
    print(f"\nSeed {seed}...")
    acc, auc, nr = train_and_evaluate(seed)
    seed_results.append({'seed': seed, 'acc': acc,
                         'auc': auc, 'neutral_recalls': nr})
    print(f"  Acc: {acc:.3f} | AUC: {auc:.3f}")
    for name, recall in nr.items():
        print(f"    Neutral — {name}: {recall:.3f}")

# --- Summary ---
print("\n" + "="*60)
print("MULTI-SEED SUMMARY")
print("="*60)

accs = [r['acc'] for r in seed_results]
aucs = [r['auc'] for r in seed_results]
print(f"\nOverall Accuracy: {np.mean(accs):.3f} +/- {np.std(accs):.3f}")
print(f"Macro AUC:        {np.mean(aucs):.3f} +/- {np.std(aucs):.3f}")

print("\nNeutral recall per group (mean +/- std):")
all_group_names = set()
for r in seed_results:
    all_group_names.update(r['neutral_recalls'].keys())

for group_name in sorted(all_group_names):
    recalls = [r['neutral_recalls'].get(group_name, 0)
               for r in seed_results
               if group_name in r['neutral_recalls']]
    if recalls:
        print(f"  {group_name}: {np.mean(recalls):.3f} "
              f"+/- {np.std(recalls):.3f}")


Seed 42...
  Acc: 0.844 | AUC: 0.971
    Neutral — Race:White/Middle Eastern: 0.824
    Neutral — Race:Black: 0.857
    Neutral — Race:Asian: 0.825
    Neutral — Gender:Male: 0.843
    Neutral — Gender:Female: 0.812
    Neutral — Age:Young: 0.834
    Neutral — Age:Middle: 0.778

Seed 67...
  Acc: 0.845 | AUC: 0.971
    Neutral — Race:White/Middle Eastern: 0.794
    Neutral — Race:Black: 0.959
    Neutral — Race:Asian: 0.841
    Neutral — Gender:Male: 0.827
    Neutral — Gender:Female: 0.804
    Neutral — Age:Young: 0.824
    Neutral — Age:Middle: 0.756

Seed 69...
  Acc: 0.846 | AUC: 0.970
    Neutral — Race:White/Middle Eastern: 0.832
    Neutral — Race:Black: 0.959
    Neutral — Race:Asian: 0.849
    Neutral — Gender:Male: 0.853
    Neutral — Gender:Female: 0.837
    Neutral — Age:Young: 0.849
    Neutral — Age:Middle: 0.811

Seed 123...
  Acc: 0.844 | AUC: 0.970
    Neutral — Race:White/Middle Eastern: 0.804
    Neutral — Race:Black: 0.878
    Neutral — Race:Asian: 0.817
    Neutra

In [5]:
def train_baseline_and_evaluate(seed):
    set_seed(seed)

    model = nn.Sequential(
        nn.Linear(768, 512),
        nn.LayerNorm(512),
        nn.GELU(),
        nn.Dropout(0.3),
        nn.Linear(512, 256),
        nn.GELU(),
        nn.Dropout(0.3),
        nn.Linear(256, 7)
    ).cuda()

    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.05)

    # Train
    for _ in range(20):
        model.train()
        for feats, labels, races, genders, ages in train_loader:
            feats, labels = feats.cuda(), labels.cuda()
            loss = criterion(model(feats), labels)
            optimizer.zero_grad(); loss.backward(); optimizer.step()

    # Evaluate
    model.eval()
    all_preds, all_labels_e = [], []
    all_races_e, all_genders_e, all_ages_e = [], [], []
    all_probs = []

    with torch.no_grad():
        for feats, labels, races, genders, ages in test_loader:
            feats = feats.cuda()
            logits = model(feats)
            probs  = torch.softmax(logits, dim=1).cpu()
            all_probs.extend(probs.tolist())
            all_preds.extend(logits.argmax(1).cpu().tolist())
            all_labels_e.extend(labels.tolist())
            all_races_e.extend(races.tolist())
            all_genders_e.extend(genders.tolist())
            all_ages_e.extend(ages.tolist())

    all_probs     = np.array(all_probs)
    all_preds     = np.array(all_preds)
    all_labels_e  = np.array(all_labels_e)
    all_races_e   = np.array(all_races_e)
    all_genders_e = np.array(all_genders_e)
    all_ages_e    = np.array(all_ages_e)

    y_bin     = label_binarize(all_labels_e, classes=list(range(7)))
    acc       = (all_preds == all_labels_e).mean()
    macro_auc = roc_auc_score(y_bin, all_probs,
                               multi_class='ovr', average='macro')

    neutral_id = 6
    neutral_recalls = {}

    for gid, gname in RACE_NAMES_4.items():
        mask = all_races_e == gid
        if mask.sum() == 0: continue
        true_n = (all_labels_e[mask] == neutral_id)
        pred_n = (all_preds[mask]    == neutral_id)
        if true_n.sum() == 0: continue
        neutral_recalls[f'Race:{gname}'] = \
            (true_n & pred_n).sum() / true_n.sum()

    for gid, gname in GENDER_MAP.items():
        mask = all_genders_e == gid
        if mask.sum() == 0: continue
        true_n = (all_labels_e[mask] == neutral_id)
        pred_n = (all_preds[mask]    == neutral_id)
        if true_n.sum() == 0: continue
        neutral_recalls[f'Gender:{gname}'] = \
            (true_n & pred_n).sum() / true_n.sum()

    for gid, gname in AGE_NAMES_3.items():
        mask = all_ages_e == gid
        if mask.sum() == 0: continue
        true_n = (all_labels_e[mask] == neutral_id)
        pred_n = (all_preds[mask]    == neutral_id)
        if true_n.sum() == 0: continue
        neutral_recalls[f'Age:{gname}'] = \
            (true_n & pred_n).sum() / true_n.sum()

    return acc, macro_auc, neutral_recalls


# Run baseline across same 9 seeds
SEEDS = [42, 67, 69, 123, 420, 456, 789, 1024, 69420]
baseline_results = []

for seed in SEEDS:
    print(f"\nBaseline Seed {seed}...")
    acc, auc, nr = train_baseline_and_evaluate(seed)
    baseline_results.append({'seed': seed, 'acc': acc,
                              'auc': auc, 'neutral_recalls': nr})
    print(f"  Acc: {acc:.3f} | AUC: {auc:.3f}")
    for name, recall in nr.items():
        print(f"    Neutral — {name}: {recall:.3f}")

print("\n" + "="*60)
print("BASELINE MULTI-SEED SUMMARY")
print("="*60)
b_accs = [r['acc'] for r in baseline_results]
b_aucs = [r['auc'] for r in baseline_results]
print(f"\nOverall Accuracy: {np.mean(b_accs):.3f} +/- {np.std(b_accs):.3f}")
print(f"Macro AUC:        {np.mean(b_aucs):.3f} +/- {np.std(b_aucs):.3f}")

print("\nNeutral recall per group (mean +/- std):")
all_group_names = set()
for r in baseline_results:
    all_group_names.update(r['neutral_recalls'].keys())
for group_name in sorted(all_group_names):
    recalls = [r['neutral_recalls'].get(group_name, 0)
               for r in baseline_results
               if group_name in r['neutral_recalls']]
    if recalls:
        print(f"  {group_name}: {np.mean(recalls):.3f} "
              f"+/- {np.std(recalls):.3f}")


Baseline Seed 42...
  Acc: 0.829 | AUC: 0.961
    Neutral — Race:White/Middle Eastern: 0.788
    Neutral — Race:Black: 0.857
    Neutral — Race:Asian: 0.778
    Neutral — Gender:Male: 0.840
    Neutral — Gender:Female: 0.749
    Neutral — Age:Young: 0.803
    Neutral — Age:Middle: 0.711

Baseline Seed 67...
  Acc: 0.832 | AUC: 0.964
    Neutral — Race:White/Middle Eastern: 0.814
    Neutral — Race:Black: 0.918
    Neutral — Race:Asian: 0.825
    Neutral — Gender:Male: 0.840
    Neutral — Gender:Female: 0.809
    Neutral — Age:Young: 0.832
    Neutral — Age:Middle: 0.767

Baseline Seed 69...
  Acc: 0.835 | AUC: 0.965
    Neutral — Race:White/Middle Eastern: 0.804
    Neutral — Race:Black: 0.776
    Neutral — Race:Asian: 0.794
    Neutral — Gender:Male: 0.812
    Neutral — Gender:Female: 0.790
    Neutral — Age:Young: 0.819
    Neutral — Age:Middle: 0.678

Baseline Seed 123...
  Acc: 0.825 | AUC: 0.962
    Neutral — Race:White/Middle Eastern: 0.818
    Neutral — Race:Black: 0.918
    Ne

In [6]:
from scipy import stats
import numpy as np

print("="*70)
print("STATISTICAL TESTS: Staged Head vs Baseline MLP")
print("="*70)

SEEDS = [42, 67, 69, 123, 420, 456, 789, 1024, 69420]

# Extract paired values (same seed = paired observation)
staged_accs   = [r['acc'] for r in seed_results]
baseline_accs = [r['acc'] for r in baseline_results]

staged_aucs   = [r['auc'] for r in seed_results]
baseline_aucs = [r['auc'] for r in baseline_results]

# All group names
all_groups = sorted(set(
    list(seed_results[0]['neutral_recalls'].keys()) +
    list(baseline_results[0]['neutral_recalls'].keys())
))

def run_tests(staged_vals, baseline_vals, metric_name):
    staged_vals   = np.array(staged_vals)
    baseline_vals = np.array(baseline_vals)
    diff = staged_vals - baseline_vals

    # Paired t-test
    t_stat, t_p = stats.ttest_rel(staged_vals, baseline_vals)

    # Wilcoxon signed-rank test
    try:
        w_stat, w_p = stats.wilcoxon(staged_vals, baseline_vals)
    except ValueError:
        w_stat, w_p = float('nan'), float('nan')

    staged_mean   = staged_vals.mean()
    baseline_mean = baseline_vals.mean()
    mean_diff     = diff.mean()

    sig_t = '***' if t_p < 0.001 else ('**' if t_p < 0.01 else
            ('*' if t_p < 0.05 else 'ns'))
    sig_w = '***' if w_p < 0.001 else ('**' if w_p < 0.01 else
            ('*' if w_p < 0.05 else 'ns'))

    print(f"\n{metric_name}")
    print(f"  Baseline:  {baseline_mean:.3f} +/- {baseline_vals.std():.3f}")
    print(f"  Staged:    {staged_mean:.3f} +/- {staged_vals.std():.3f}")
    print(f"  Mean diff: {mean_diff:+.3f}")
    print(f"  Paired t-test:  t={t_stat:.3f}, p={t_p:.4f} {sig_t}")
    print(f"  Wilcoxon:       W={w_stat:.3f}, p={w_p:.4f} {sig_w}")

    return {
        'metric': metric_name,
        'baseline_mean': baseline_mean,
        'staged_mean': staged_mean,
        'mean_diff': mean_diff,
        't_stat': t_stat, 't_p': t_p,
        'w_stat': w_stat, 'w_p': w_p,
        'sig_t': sig_t, 'sig_w': sig_w
    }

all_test_results = []

# Overall accuracy
print("\n--- OVERALL METRICS ---")
all_test_results.append(
    run_tests(staged_accs, baseline_accs, "Overall Accuracy"))
all_test_results.append(
    run_tests(staged_aucs, baseline_aucs, "Macro AUC"))

# Per-group Neutral recall
print("\n--- NEUTRAL RECALL PER GROUP ---")
for group in all_groups:
    staged_vals   = [r['neutral_recalls'].get(group, np.nan)
                     for r in seed_results]
    baseline_vals = [r['neutral_recalls'].get(group, np.nan)
                     for r in baseline_results]

    # Skip if any NaN
    pairs = [(s, b) for s, b in zip(staged_vals, baseline_vals)
             if not np.isnan(s) and not np.isnan(b)]
    if len(pairs) < 5:
        print(f"\n{group}: insufficient data")
        continue

    s_vals = [p[0] for p in pairs]
    b_vals = [p[1] for p in pairs]
    all_test_results.append(run_tests(s_vals, b_vals, group))

# --- Summary table ---
print("\n\n" + "="*70)
print("SUMMARY TABLE")
print("="*70)
print(f"\n{'Metric':<35} {'Baseline':>9} {'Staged':>9} "
      f"{'Diff':>7} {'t-test':>8} {'Wilcoxon':>9}")
print("-"*70)
for r in all_test_results:
    print(f"{r['metric']:<35} "
          f"{r['baseline_mean']:>9.3f} "
          f"{r['staged_mean']:>9.3f} "
          f"{r['mean_diff']:>+7.3f} "
          f"{r['sig_t']:>8} "
          f"{r['sig_w']:>9}")

print("\nSignificance: *** p<0.001  ** p<0.01  * p<0.05  ns = not significant")

STATISTICAL TESTS: Staged Head vs Baseline MLP

--- OVERALL METRICS ---

Overall Accuracy
  Baseline:  0.836 +/- 0.006
  Staged:    0.844 +/- 0.002
  Mean diff: +0.009
  Paired t-test:  t=4.147, p=0.0032 **
  Wilcoxon:       W=2.000, p=0.0117 *

Macro AUC
  Baseline:  0.964 +/- 0.002
  Staged:    0.970 +/- 0.001
  Mean diff: +0.006
  Paired t-test:  t=7.137, p=0.0001 ***
  Wilcoxon:       W=0.000, p=0.0039 **

--- NEUTRAL RECALL PER GROUP ---

Age:Middle
  Baseline:  0.740 +/- 0.037
  Staged:    0.757 +/- 0.033
  Mean diff: +0.017
  Paired t-test:  t=0.786, p=0.4547 ns
  Wilcoxon:       W=15.000, p=0.7188 ns

Age:Young
  Baseline:  0.838 +/- 0.023
  Staged:    0.832 +/- 0.012
  Mean diff: -0.006
  Paired t-test:  t=-0.535, p=0.6073 ns
  Wilcoxon:       W=21.000, p=0.9102 ns

Gender:Female
  Baseline:  0.815 +/- 0.034
  Staged:    0.812 +/- 0.018
  Mean diff: -0.003
  Paired t-test:  t=-0.204, p=0.8432 ns
  Wilcoxon:       W=20.500, p=0.8398 ns

Gender:Male
  Baseline:  0.836 +/- 0.024


In [7]:
import os

DEMO_CACHE = '/content/drive/MyDrive/Colab Notebooks/demographic_classifier'

files = os.listdir(DEMO_CACHE)
for f in sorted(files):
    print(f)

affectnet_train_autolabeled.pt
affectnet_val_autolabeled.pt
confidence_distribution.png
demo_clf_race_cm.png
demographic_classifier.pt
demographic_classifier_4class.pt
demographic_classifier_4race_3age.pt
fairface_train_4class.pt
fairface_train_features.pt
fairface_val_4class.pt
fairface_val_features.pt
staged_emotion_head.pt
staged_head_cm.png
staged_loss_curves.png
staged_loss_curves_combined_val.png
staged_loss_curves_normalized.png
unified_emotion_head.pt
unified_head_cm.png
unified_train.pt


In [8]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import label_binarize
from collections import Counter
import numpy as np

# ================================================================
# LOAD ORIGINAL FAIRFACE FEATURES (7-class race, 9-bin age)
# ================================================================
train_ff_orig = torch.load(f'{DEMO_CACHE}/fairface_train_features.pt',
                            weights_only=False)
val_ff_orig   = torch.load(f'{DEMO_CACHE}/fairface_val_features.pt',
                            weights_only=False)

print("Original FairFace schema:")
print(f"  Train: {train_ff_orig['features'].shape}")
print(f"  Race classes: {train_ff_orig['races'].max().item() + 1}")
print(f"  Age classes:  {train_ff_orig['ages'].max().item() + 1}")
print(f"  Gender classes: {train_ff_orig['genders'].max().item() + 1}")

# ================================================================
# SCHEME A — ORIGINAL: 7-class race, 9-bin age, 2-class gender
# ================================================================
RACE_NAMES_7 = {
    0: 'East Asian', 1: 'Indian', 2: 'Black',
    3: 'White', 4: 'Middle Eastern',
    5: 'Latino', 6: 'SE Asian'
}
AGE_NAMES_9 = {
    0: '0-2', 1: '3-9', 2: '10-19', 3: '20-29', 4: '30-39',
    5: '40-49', 6: '50-59', 7: '60-69', 8: '70+'
}

# ================================================================
# SCHEME B — CONSOLIDATED: 4-class race, 3-bin age, 2-class gender
# ================================================================
FAIRFACE_TO_4CLASS = {
    3: 0, 4: 0,  # White + ME → White/ME
    2: 1,        # Black
    0: 2, 6: 2,  # East + SE Asian → Asian
    1: 3, 5: 3,  # Indian + Latino → Indian/Latino
}
AGE_TO_3CLASS = {0:0,1:0,2:0, 3:1,4:1,5:1, 6:2,7:2,8:2}

def remap_tensor(tensor, mapping):
    return torch.tensor(
        [mapping[v.item()] for v in tensor], dtype=torch.long)

# Remap for scheme B
train_ff_4 = dict(train_ff_orig)
train_ff_4['races'] = remap_tensor(train_ff_orig['races'], FAIRFACE_TO_4CLASS)
train_ff_4['ages']  = remap_tensor(train_ff_orig['ages'],  AGE_TO_3CLASS)

val_ff_4 = dict(val_ff_orig)
val_ff_4['races'] = remap_tensor(val_ff_orig['races'], FAIRFACE_TO_4CLASS)
val_ff_4['ages']  = remap_tensor(val_ff_orig['ages'],  AGE_TO_3CLASS)

# ================================================================
# DEMOGRAPHIC CLASSIFIER — generic class
# ================================================================
class DemoClf(nn.Module):
    def __init__(self, num_races, num_ages, num_genders=2,
                 feature_dim=768, hidden_dim=512, dropout=0.3):
        super().__init__()
        self.trunk = nn.Sequential(
            nn.Linear(feature_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 256),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.race_head   = nn.Linear(256, num_races)
        self.gender_head = nn.Linear(256, num_genders)
        self.age_head    = nn.Linear(256, num_ages)

    def forward(self, x):
        s = self.trunk(x)
        return self.race_head(s), self.gender_head(s), self.age_head(s)


def train_demo_clf(train_cache, val_cache, num_races, num_ages,
                   num_epochs=20, batch_size=512):
    train_loader = DataLoader(
        TensorDataset(train_cache['features'], train_cache['races'],
                      train_cache['genders'], train_cache['ages']),
        batch_size=batch_size, shuffle=True
    )
    val_loader = DataLoader(
        TensorDataset(val_cache['features'], val_cache['races'],
                      val_cache['genders'], val_cache['ages']),
        batch_size=batch_size, shuffle=False
    )

    model = DemoClf(num_races=num_races, num_ages=num_ages).cuda()
    opt   = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.05)
    crit  = nn.CrossEntropyLoss()

    for epoch in range(num_epochs):
        model.train()
        for feats, races, genders, ages in train_loader:
            feats, races = feats.cuda(), races.cuda()
            genders, ages = genders.cuda(), ages.cuda()
            r_l, g_l, a_l = model(feats)
            loss = crit(r_l, races) + crit(g_l, genders) + crit(a_l, ages)
            opt.zero_grad(); loss.backward(); opt.step()

    # Evaluate
    model.eval()
    r_correct = g_correct = a_correct = total = 0
    r_per_class = {i: {'correct': 0, 'total': 0} for i in range(num_races)}
    a_per_class = {i: {'correct': 0, 'total': 0} for i in range(num_ages)}

    with torch.no_grad():
        for feats, races, genders, ages in val_loader:
            feats = feats.cuda()
            r_l, g_l, a_l = model(feats)
            r_pred = r_l.argmax(1).cpu()
            g_pred = g_l.argmax(1).cpu()
            a_pred = a_l.argmax(1).cpu()

            r_correct += (r_pred == races).sum().item()
            g_correct += (g_pred == genders).sum().item()
            a_correct += (a_pred == ages).sum().item()
            total += feats.size(0)

            for i in range(num_races):
                mask = races == i
                r_per_class[i]['total']   += mask.sum().item()
                r_per_class[i]['correct'] += (r_pred[mask] == i).sum().item()

            for i in range(num_ages):
                mask = ages == i
                a_per_class[i]['total']   += mask.sum().item()
                a_per_class[i]['correct'] += (a_pred[mask] == i).sum().item()

    return {
        'race_acc':   r_correct / total,
        'gender_acc': g_correct / total,
        'age_acc':    a_correct / total,
        'r_per_class': r_per_class,
        'a_per_class': a_per_class,
        'model': model
    }


# ================================================================
# TRAIN SCHEME A — 7-class race, 9-bin age
# ================================================================
print("\n" + "="*60)
print("SCHEME A: 7-class race, 9-bin age")
print("="*60)
results_A = train_demo_clf(train_ff_orig, val_ff_orig,
                           num_races=7, num_ages=9, num_epochs=20)
print(f"Overall Race Acc:   {results_A['race_acc']:.3f}")
print(f"Overall Gender Acc: {results_A['gender_acc']:.3f}")
print(f"Overall Age Acc:    {results_A['age_acc']:.3f}")
print("\nPer-race recall:")
for i, name in RACE_NAMES_7.items():
    d = results_A['r_per_class'][i]
    if d['total'] > 0:
        print(f"  {name:<18}: {d['correct']/d['total']:.3f}  (n={d['total']})")
print("\nPer-age recall:")
for i, name in AGE_NAMES_9.items():
    d = results_A['a_per_class'][i]
    if d['total'] > 0:
        print(f"  {name:<8}: {d['correct']/d['total']:.3f}  (n={d['total']})")


# ================================================================
# TRAIN SCHEME B — 4-class race, 3-bin age
# ================================================================
print("\n" + "="*60)
print("SCHEME B: 4-class race, 3-bin age")
print("="*60)
results_B = train_demo_clf(train_ff_4, val_ff_4,
                           num_races=4, num_ages=3, num_epochs=20)

RACE_NAMES_4_LIST = {0: 'White/Middle Eastern', 1: 'Black',
                     2: 'Asian', 3: 'Indian/Latino'}
AGE_NAMES_3_LIST  = {0: 'Young (0-19)', 1: 'Middle (20-49)',
                     2: 'Old (50+)'}

print(f"Overall Race Acc:   {results_B['race_acc']:.3f}")
print(f"Overall Gender Acc: {results_B['gender_acc']:.3f}")
print(f"Overall Age Acc:    {results_B['age_acc']:.3f}")
print("\nPer-race recall:")
for i, name in RACE_NAMES_4_LIST.items():
    d = results_B['r_per_class'][i]
    if d['total'] > 0:
        print(f"  {name:<22}: {d['correct']/d['total']:.3f}  (n={d['total']})")
print("\nPer-age recall:")
for i, name in AGE_NAMES_3_LIST.items():
    d = results_B['a_per_class'][i]
    if d['total'] > 0:
        print(f"  {name:<18}: {d['correct']/d['total']:.3f}  (n={d['total']})")


# ================================================================
# COMPARISON SUMMARY
# ================================================================
print("\n" + "="*60)
print("COMPARISON SUMMARY")
print("="*60)
print(f"\n{'Metric':<25} {'Scheme A':>10} {'Scheme B':>10} {'Delta':>8}")
print("-"*55)
print(f"{'Race accuracy':<25} {results_A['race_acc']:>10.3f} "
      f"{results_B['race_acc']:>10.3f} "
      f"{results_B['race_acc']-results_A['race_acc']:>+8.3f}")
print(f"{'Gender accuracy':<25} {results_A['gender_acc']:>10.3f} "
      f"{results_B['gender_acc']:>10.3f} "
      f"{results_B['gender_acc']-results_A['gender_acc']:>+8.3f}")
print(f"{'Age accuracy':<25} {results_A['age_acc']:>10.3f} "
      f"{results_B['age_acc']:>10.3f} "
      f"{results_B['age_acc']-results_A['age_acc']:>+8.3f}")

Original FairFace schema:
  Train: torch.Size([86744, 768])
  Race classes: 7
  Age classes:  9
  Gender classes: 2

SCHEME A: 7-class race, 9-bin age
Overall Race Acc:   0.720
Overall Gender Acc: 0.946
Overall Age Acc:    0.602

Per-race recall:
  East Asian        : 0.808  (n=1550)
  Indian            : 0.738  (n=1516)
  Black             : 0.879  (n=1556)
  White             : 0.783  (n=2085)
  Middle Eastern    : 0.646  (n=1209)
  Latino            : 0.532  (n=1623)
  SE Asian          : 0.616  (n=1415)

Per-age recall:
  0-2     : 0.693  (n=199)
  3-9     : 0.854  (n=1356)
  10-19   : 0.404  (n=1181)
  20-29   : 0.727  (n=3300)
  30-39   : 0.522  (n=2330)
  40-49   : 0.442  (n=1353)
  50-59   : 0.501  (n=796)
  60-69   : 0.489  (n=321)
  70+     : 0.441  (n=118)

SCHEME B: 4-class race, 3-bin age
Overall Race Acc:   0.840
Overall Gender Acc: 0.950
Overall Age Acc:    0.876

Per-race recall:
  White/Middle Eastern  : 0.836  (n=3294)
  Black                 : 0.860  (n=1556)
  Asian

In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import label_binarize
from collections import Counter
import numpy as np

# ================================================================
# STEP 1: Auto-label AffectNet with ORIGINAL schema (7-race, 9-age)
# ================================================================
# Load AffectNet cache (already has auto-labels from 4-class classifier)
# We need to re-label using the 7-class demographic classifier
an_train_cache_orig = torch.load(
    f'{DEMO_CACHE}/affectnet_train_autolabeled.pt',
    weights_only=False
)

# Load the original 7-class demographic classifier (trained above as results_A)
demo_clf_7 = results_A['model']
demo_clf_7.eval()

# Re-predict demographics using 7-class model on AffectNet features
import torch.nn.functional as F

an_feats = an_train_cache_orig['features']
batch_size = 512
all_r_pred, all_g_pred, all_a_pred, all_r_conf = [], [], [], []

with torch.no_grad():
    for i in range(0, len(an_feats), batch_size):
        batch = an_feats[i:i+batch_size].cuda()
        r_l, g_l, a_l = demo_clf_7(batch)
        r_probs = F.softmax(r_l, dim=1).cpu()
        r_conf, r_pred = r_probs.max(dim=1)
        _, g_pred = F.softmax(g_l, dim=1).cpu().max(dim=1)
        _, a_pred = F.softmax(a_l, dim=1).cpu().max(dim=1)
        all_r_pred.extend(r_pred.tolist())
        all_g_pred.extend(g_pred.tolist())
        all_a_pred.extend(a_pred.tolist())
        all_r_conf.extend(r_conf.tolist())

all_r_pred = torch.tensor(all_r_pred)
all_g_pred = torch.tensor(all_g_pred)
all_a_pred = torch.tensor(all_a_pred)
all_r_conf = torch.tensor(all_r_conf)

# Apply confidence threshold
conf_mask = all_r_conf >= 0.70
print(f"AffectNet confident samples (7-class): {conf_mask.sum().item()}")

# Remap AffectNet emotion labels
AFFECTNET_TO_RAFDB = {0:5,1:2,2:2,3:1,4:3,5:6,6:4,7:0}
an_raw_labels = an_train_cache_orig['labels'][conf_mask]
an_labels_orig = torch.tensor(
    [AFFECTNET_TO_RAFDB[l.item()] for l in an_raw_labels],
    dtype=torch.long
)

# ================================================================
# STEP 2: Build unified train with ORIGINAL schema
# ================================================================

# Load RAF-DB with original race schema (3-class, maps to 7-class equivalent)
# RAF-DB only has 3 race groups — map to 7-class
# White→3, Black→2, Asian→0 (East Asian as proxy)
RAFDB_TO_7CLASS = {0: 3, 1: 2, 2: 0}

train_raf_orig = torch.load(
    f'{RAFDB_BASE_CACHE}/train_features.pt', weights_only=False)

# Remap RAF-DB races to 7-class
raf_races_7 = torch.tensor(
    [RAFDB_TO_7CLASS[r.item()] for r in train_raf_orig['races']],
    dtype=torch.long
)

# RAF-DB ages stay as original 5-class → no consolidation
# But we need 9-bin — use original ages directly (already 0-4 range)
# Map RAF-DB 5-bin to 9-bin approximately
RAFDB_AGE_TO_9 = {0:0, 1:2, 2:3, 3:6, 4:8}
raf_ages_9 = torch.tensor(
    [RAFDB_AGE_TO_9[a.item()] for a in train_raf_orig['ages']],
    dtype=torch.long
)

unified_orig = {
    'features': torch.cat([
        train_raf_orig['features'],
        an_train_cache_orig['features'][conf_mask]
    ]),
    'labels': torch.cat([
        train_raf_orig['labels'],
        an_labels_orig
    ]),
    'races': torch.cat([
        raf_races_7,
        all_r_pred[conf_mask]
    ]),
    'genders': torch.cat([
        train_raf_orig['genders'].clamp(0,1),
        all_g_pred[conf_mask]
    ]),
    'ages': torch.cat([
        raf_ages_9,
        all_a_pred[conf_mask]
    ]),
}

print(f"Unified train (original schema): {len(unified_orig['features'])} samples")
print(f"Race max: {unified_orig['races'].max().item()} (expect 6)")
print(f"Age max:  {unified_orig['ages'].max().item()} (expect 8)")

# ================================================================
# STEP 3: RAF-DB test set with original schema
# ================================================================
test_raf_orig = torch.load(
    f'{RAFDB_BASE_CACHE}/test_features.pt', weights_only=False)

raf_test_races_7 = torch.tensor(
    [RAFDB_TO_7CLASS[r.item()] for r in test_raf_orig['races']],
    dtype=torch.long
)
raf_test_ages_9 = torch.tensor(
    [RAFDB_AGE_TO_9[a.item()] for a in test_raf_orig['ages']],
    dtype=torch.long
)

test_orig = {
    'features': test_raf_orig['features'],
    'labels':   test_raf_orig['labels'],
    'races':    raf_test_races_7,
    'genders':  test_raf_orig['genders'].clamp(0,1),
    'ages':     raf_test_ages_9,
}

# ================================================================
# STEP 4: Train staged head with ORIGINAL schema
# ================================================================
class StagedEmotionHeadOrig(nn.Module):
    def __init__(self, feature_dim=768,
                 num_races=7, num_genders=2, num_ages=9,
                 embed_dim=32, hidden_dim=512,
                 num_emotions=7, dropout=0.3):
        super().__init__()
        self.race_emb   = nn.Embedding(num_races,   embed_dim)
        self.gender_emb = nn.Embedding(num_genders, embed_dim)
        self.age_emb    = nn.Embedding(num_ages,    embed_dim)
        self.race_aux   = nn.Linear(embed_dim, num_races)
        self.gender_aux = nn.Linear(embed_dim, num_genders)
        self.age_aux    = nn.Linear(embed_dim, num_ages)
        combined_dim = feature_dim + embed_dim * 3
        self.classifier = nn.Sequential(
            nn.Linear(combined_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, num_emotions)
        )

    def forward(self, feats, races, genders, ages):
        combined = torch.cat([
            feats,
            self.race_emb(races),
            self.gender_emb(genders),
            self.age_emb(ages)
        ], dim=-1)
        return self.classifier(combined)

    def forward_demo(self, races, genders, ages):
        return (self.race_aux(self.race_emb(races)),
                self.gender_aux(self.gender_emb(genders)),
                self.age_aux(self.age_emb(ages)))


counts_orig  = Counter(unified_orig['labels'].tolist())
total_orig   = sum(counts_orig.values())
weights_orig = torch.tensor(
    [total_orig/counts_orig[i] for i in range(7)],
    dtype=torch.float
).cuda()
weights_orig = weights_orig / weights_orig.sum()

train_loader_orig = DataLoader(
    TensorDataset(
        unified_orig['features'], unified_orig['labels'],
        unified_orig['races'],    unified_orig['genders'],
        unified_orig['ages']
    ),
    batch_size=256, shuffle=True
)
test_loader_orig = DataLoader(
    TensorDataset(
        test_orig['features'], test_orig['labels'],
        test_orig['races'],    test_orig['genders'],
        test_orig['ages']
    ),
    batch_size=256, shuffle=False
)

print("\nTraining staged head with ORIGINAL schema (7-race, 9-age)...")
set_seed(42)
model_orig = StagedEmotionHeadOrig().cuda()
criterion_emotion_orig = nn.CrossEntropyLoss(weight=weights_orig)
criterion_demo_orig    = nn.CrossEntropyLoss()

# Stage 1
for param in model_orig.classifier.parameters():
    param.requires_grad = False
stage1_params = (
    list(model_orig.race_emb.parameters()) +
    list(model_orig.gender_emb.parameters()) +
    list(model_orig.age_emb.parameters()) +
    list(model_orig.race_aux.parameters()) +
    list(model_orig.gender_aux.parameters()) +
    list(model_orig.age_aux.parameters())
)
opt1 = optim.AdamW(stage1_params, lr=1e-3, weight_decay=0.05)
for epoch in range(15):
    model_orig.train()
    total_loss = 0
    for feats, labels, races, genders, ages in train_loader_orig:
        races, genders, ages = races.cuda(), genders.cuda(), ages.cuda()
        r_l, g_l, a_l = model_orig.forward_demo(races, genders, ages)
        loss = (criterion_demo_orig(r_l, races) +
                criterion_demo_orig(g_l, genders) +
                criterion_demo_orig(a_l, ages))
        opt1.zero_grad(); loss.backward(); opt1.step()
        total_loss += loss.item()
    print(f"  Stage 1 Epoch {epoch+1:02d}/15 | Loss: {total_loss/len(train_loader_orig):.4f}")

# Stage 2
for param in model_orig.race_emb.parameters():   param.requires_grad = False
for param in model_orig.gender_emb.parameters(): param.requires_grad = False
for param in model_orig.age_emb.parameters():    param.requires_grad = False
for param in model_orig.race_aux.parameters():   param.requires_grad = False
for param in model_orig.gender_aux.parameters(): param.requires_grad = False
for param in model_orig.age_aux.parameters():    param.requires_grad = False
for param in model_orig.classifier.parameters(): param.requires_grad = True
opt2 = optim.AdamW(model_orig.classifier.parameters(),
                   lr=1e-3, weight_decay=0.05)
for epoch in range(40):
    model_orig.train()
    total_loss, correct, total = 0, 0, 0
    for feats, labels, races, genders, ages in train_loader_orig:
        feats, labels = feats.cuda(), labels.cuda()
        races, genders, ages = races.cuda(), genders.cuda(), ages.cuda()
        logits = model_orig(feats, races, genders, ages)
        loss   = criterion_emotion_orig(logits, labels)
        opt2.zero_grad(); loss.backward(); opt2.step()
        total_loss += loss.item()
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += labels.size(0)
    if (epoch+1) % 10 == 0:
        print(f"  Stage 2 Epoch {epoch+1:02d}/40 | "
              f"Loss: {total_loss/len(train_loader_orig):.4f} | "
              f"Train: {correct/total:.3f}")

# Stage 3
for param in model_orig.parameters():
    param.requires_grad = True
opt3 = optim.AdamW(model_orig.parameters(), lr=1e-4, weight_decay=0.05)
for epoch in range(20):
    model_orig.train()
    total_loss, correct, total = 0, 0, 0
    for feats, labels, races, genders, ages in train_loader_orig:
        feats, labels = feats.cuda(), labels.cuda()
        races, genders, ages = races.cuda(), genders.cuda(), ages.cuda()
        logits = model_orig(feats, races, genders, ages)
        loss   = criterion_emotion_orig(logits, labels)
        opt3.zero_grad(); loss.backward(); opt3.step()
        total_loss += loss.item()
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += labels.size(0)
    if (epoch+1) % 5 == 0:
        print(f"  Stage 3 Epoch {epoch+1:02d}/20 | "
              f"Loss: {total_loss/len(train_loader_orig):.4f} | "
              f"Train: {correct/total:.3f}")

# ================================================================
# STEP 5: Evaluate original schema model
# ================================================================
model_orig.eval()
all_preds, all_labels_e, all_probs = [], [], []
all_races_e, all_genders_e, all_ages_e = [], [], []

with torch.no_grad():
    for feats, labels, races, genders, ages in test_loader_orig:
        feats = feats.cuda()
        races, genders, ages = races.cuda(), genders.cuda(), ages.cuda()
        logits = model_orig(feats, races, genders, ages)
        probs  = torch.softmax(logits, dim=1).cpu()
        all_probs.extend(probs.tolist())
        all_preds.extend(logits.argmax(1).cpu().tolist())
        all_labels_e.extend(labels.tolist())
        all_races_e.extend(races.cpu().tolist())
        all_genders_e.extend(genders.cpu().tolist())
        all_ages_e.extend(ages.cpu().tolist())

all_probs    = np.array(all_probs)
all_preds    = np.array(all_preds)
all_labels_e = np.array(all_labels_e)
all_races_e  = np.array(all_races_e)
all_genders_e = np.array(all_genders_e)
all_ages_e   = np.array(all_ages_e)

y_bin     = label_binarize(all_labels_e, classes=list(range(7)))
acc_orig  = (all_preds == all_labels_e).mean()
auc_orig  = roc_auc_score(y_bin, all_probs,
                           multi_class='ovr', average='macro')

neutral_id = 6
print("\n" + "="*60)
print("RESULTS: Original Schema (7-race, 9-age)")
print("="*60)
print(f"Overall Accuracy: {acc_orig:.3f}")
print(f"Macro AUC:        {auc_orig:.3f}")

print("\nNeutral recall per group:")
RACE_7_MAP = {0:'East Asian',1:'Indian',2:'Black',
              3:'White',4:'Middle Eastern',5:'Latino',6:'SE Asian'}
for rid, rname in RACE_7_MAP.items():
    mask = all_races_e == rid
    if mask.sum() == 0: continue
    true_n = (all_labels_e[mask] == neutral_id)
    pred_n = (all_preds[mask]    == neutral_id)
    if true_n.sum() == 0: continue
    print(f"  Race:{rname}: {(true_n&pred_n).sum()/true_n.sum():.3f} (n={true_n.sum()})")

for gid, gname in GENDER_MAP.items():
    mask = all_genders_e == gid
    if mask.sum() == 0: continue
    true_n = (all_labels_e[mask] == neutral_id)
    pred_n = (all_preds[mask]    == neutral_id)
    if true_n.sum() == 0: continue
    print(f"  Gender:{gname}: {(true_n&pred_n).sum()/true_n.sum():.3f}")

AGE_9_MAP = {0:'0-2',1:'3-9',2:'10-19',3:'20-29',4:'30-39',
             5:'40-49',6:'50-59',7:'60-69',8:'70+'}
for aid, aname in AGE_9_MAP.items():
    mask = all_ages_e == aid
    if mask.sum() == 0: continue
    true_n = (all_labels_e[mask] == neutral_id)
    pred_n = (all_preds[mask]    == neutral_id)
    if true_n.sum() == 0: continue
    print(f"  Age:{aname}: {(true_n&pred_n).sum()/true_n.sum():.3f} (n={true_n.sum()})")

# ================================================================
# STEP 6: Final comparison
# ================================================================
print("\n" + "="*60)
print("PRE vs POST CONSOLIDATION — EMOTION HEAD COMPARISON")
print("="*60)
print(f"\n{'Metric':<35} {'Original':>10} {'Consolidated':>13} {'Delta':>8}")
print("-"*68)
acc_consol = np.mean([r['acc'] for r in seed_results])
auc_consol = np.mean([r['auc'] for r in seed_results])
print(f"{'Overall Accuracy':<35} {acc_orig:>10.3f} {acc_consol:>13.3f} "
      f"{acc_consol-acc_orig:>+8.3f}")
print(f"{'Macro AUC':<35} {auc_orig:>10.3f} {auc_consol:>13.3f} "
      f"{auc_consol-auc_orig:>+8.3f}")

AffectNet confident samples (7-class): 9121
Unified train (original schema): 21392 samples
Race max: 6 (expect 6)
Age max:  8 (expect 8)

Training staged head with ORIGINAL schema (7-race, 9-age)...
  Stage 1 Epoch 01/15 | Loss: 3.1711
  Stage 1 Epoch 02/15 | Loss: 0.8679
  Stage 1 Epoch 03/15 | Loss: 0.3109
  Stage 1 Epoch 04/15 | Loss: 0.1574
  Stage 1 Epoch 05/15 | Loss: 0.0967
  Stage 1 Epoch 06/15 | Loss: 0.0665
  Stage 1 Epoch 07/15 | Loss: 0.0489
  Stage 1 Epoch 08/15 | Loss: 0.0377
  Stage 1 Epoch 09/15 | Loss: 0.0301
  Stage 1 Epoch 10/15 | Loss: 0.0246
  Stage 1 Epoch 11/15 | Loss: 0.0206
  Stage 1 Epoch 12/15 | Loss: 0.0175
  Stage 1 Epoch 13/15 | Loss: 0.0151
  Stage 1 Epoch 14/15 | Loss: 0.0132
  Stage 1 Epoch 15/15 | Loss: 0.0116
  Stage 2 Epoch 10/40 | Loss: 0.2972 | Train: 0.893
  Stage 2 Epoch 20/40 | Loss: 0.1575 | Train: 0.943
  Stage 2 Epoch 30/40 | Loss: 0.1201 | Train: 0.958
  Stage 2 Epoch 40/40 | Loss: 0.0964 | Train: 0.967
  Stage 3 Epoch 05/20 | Loss: 0.0400 |

In [10]:
# K-fold results from previous run (5-fold, staged head)
fold_results = [
    {'acc': 0.760, 'auc': 0.954, 'neutral_recalls': {
        'White/Middle Eastern': 0.748, 'Black': 0.815,
        'Asian': 0.794, 'Indian/Latino': 0.250}},
    {'acc': 0.754, 'auc': 0.955, 'neutral_recalls': {
        'White/Middle Eastern': 0.695, 'Black': 0.786,
        'Asian': 0.790, 'Indian/Latino': 0.474}},
    {'acc': 0.761, 'auc': 0.957, 'neutral_recalls': {
        'White/Middle Eastern': 0.765, 'Black': 0.722,
        'Asian': 0.810, 'Indian/Latino': 0.364}},
    {'acc': 0.754, 'auc': 0.956, 'neutral_recalls': {
        'White/Middle Eastern': 0.773, 'Black': 0.753,
        'Asian': 0.765, 'Indian/Latino': 0.267}},
    {'acc': 0.772, 'auc': 0.957, 'neutral_recalls': {
        'White/Middle Eastern': 0.753, 'Black': 0.811,
        'Asian': 0.771, 'Indian/Latino': 0.278}},
]

print("K-fold results loaded from previous run.")
print(f"Mean Acc: {np.mean([r['acc'] for r in fold_results]):.3f} +/- "
      f"{np.std([r['acc'] for r in fold_results]):.3f}")

K-fold results loaded from previous run.
Mean Acc: 0.760 +/- 0.007


In [11]:
print("="*70)
print("TABLE: MULTI-SEED vs K-FOLD COMPARISON")
print("="*70)

# Multi-seed results (staged head, 9 seeds)
staged_accs = [r['acc'] for r in seed_results]
staged_aucs = [r['auc'] for r in seed_results]

# K-fold results from earlier
kfold_accs = [r['acc'] for r in fold_results]
kfold_aucs = [r['auc'] for r in fold_results]

print(f"\n{'Metric':<25} {'Multi-Seed (n=9)':>18} {'K-Fold (k=5)':>15}")
print("-"*60)
print(f"{'Overall Accuracy':<25} "
      f"{np.mean(staged_accs):.3f} +/- {np.std(staged_accs):.3f}   "
      f"{np.mean(kfold_accs):.3f} +/- {np.std(kfold_accs):.3f}")
print(f"{'Macro AUC':<25} "
      f"{np.mean(staged_aucs):.3f} +/- {np.std(staged_aucs):.3f}   "
      f"{np.mean(kfold_aucs):.3f} +/- {np.std(kfold_aucs):.3f}")

print("\nNeutral recall per group:")
kfold_group_names = set()
for r in fold_results:
    kfold_group_names.update(r['neutral_recalls'].keys())

# Multi-seed groups
ms_groups = set()
for r in seed_results:
    ms_groups.update(r['neutral_recalls'].keys())

all_groups = sorted(ms_groups)
for group in all_groups:
    ms_recalls = [r['neutral_recalls'].get(group, np.nan)
                  for r in seed_results
                  if group in r['neutral_recalls']]
    # K-fold uses race names directly without prefix
    kf_key = group.replace('Race:', '')
    kf_recalls = [r['neutral_recalls'].get(kf_key, np.nan)
                  for r in fold_results
                  if kf_key in r['neutral_recalls']]

    ms_str = (f"{np.mean(ms_recalls):.3f} +/- {np.std(ms_recalls):.3f}"
              if ms_recalls else "N/A")
    kf_str = (f"{np.mean(kf_recalls):.3f} +/- {np.std(kf_recalls):.3f}"
              if kf_recalls else "N/A")
    print(f"  {group:<33} {ms_str:>20} {kf_str:>18}")

print(f"\nNote: Multi-seed evaluates on RAF-DB test set (fixed).")
print(f"      K-fold evaluates on rotating val folds (RAF-DB + AffectNet).")
print(f"      Lower k-fold accuracy reflects AffectNet auto-label noise.")


print("\n\n" + "="*70)
print("FINAL CONSOLIDATED RESULTS TABLE")
print("="*70)

print(f"""
Staged Head (RAF-DB + AffectNet, 9 seeds):
  Overall Accuracy : {np.mean(staged_accs):.3f} +/- {np.std(staged_accs):.3f}
  Macro AUC        : {np.mean(staged_aucs):.3f} +/- {np.std(staged_aucs):.3f}

Baseline MLP (9 seeds):
  Overall Accuracy : {np.mean([r['acc'] for r in baseline_results]):.3f} +/- {np.std([r['acc'] for r in baseline_results]):.3f}
  Macro AUC        : {np.mean([r['auc'] for r in baseline_results]):.3f} +/- {np.std([r['auc'] for r in baseline_results]):.3f}

Statistical significance (staged vs baseline):
  Accuracy: p=0.0032 (**)
  AUC:      p=0.0001 (***)
""")

print(f"{'Group':<35} {'Baseline':>12} {'Staged':>12} {'Delta':>8}")
print("-"*70)

all_groups_final = sorted(set(
    list(seed_results[0]['neutral_recalls'].keys()) +
    list(baseline_results[0]['neutral_recalls'].keys())
))

for group in all_groups_final:
    s_recalls = [r['neutral_recalls'].get(group, np.nan)
                 for r in seed_results
                 if group in r['neutral_recalls']]
    b_recalls = [r['neutral_recalls'].get(group, np.nan)
                 for r in baseline_results
                 if group in r['neutral_recalls']]
    if not s_recalls or not b_recalls:
        continue
    s_mean = np.mean(s_recalls)
    b_mean = np.mean(b_recalls)
    s_std  = np.std(s_recalls)
    b_std  = np.std(b_recalls)
    delta  = s_mean - b_mean
    print(f"  {group:<33} "
          f"{b_mean:.3f}+/-{b_std:.3f}   "
          f"{s_mean:.3f}+/-{s_std:.3f}   "
          f"{delta:>+7.3f}")

print("\nPre vs Post Consolidation (emotion head, seed=42):")
print(f"  Original  (7-race, 9-age): Acc={acc_orig:.3f}, AUC={auc_orig:.3f}")
print(f"  Consolidated (4-race, 3-age): Acc={acc_consol:.3f}, AUC={auc_consol:.3f}")
print(f"  Delta: Acc={acc_consol-acc_orig:+.3f}, AUC={auc_consol-auc_orig:+.3f}")

TABLE: MULTI-SEED vs K-FOLD COMPARISON

Metric                      Multi-Seed (n=9)    K-Fold (k=5)
------------------------------------------------------------
Overall Accuracy          0.844 +/- 0.002   0.760 +/- 0.007
Macro AUC                 0.970 +/- 0.001   0.956 +/- 0.001

Neutral recall per group:
  Age:Middle                             0.757 +/- 0.033                N/A
  Age:Young                              0.832 +/- 0.012                N/A
  Gender:Female                          0.812 +/- 0.018                N/A
  Gender:Male                            0.833 +/- 0.018                N/A
  Race:Asian                             0.840 +/- 0.021    0.786 +/- 0.016
  Race:Black                             0.916 +/- 0.033    0.777 +/- 0.035
  Race:White/Middle Eastern              0.808 +/- 0.017    0.747 +/- 0.027

Note: Multi-seed evaluates on RAF-DB test set (fixed).
      K-fold evaluates on rotating val folds (RAF-DB + AffectNet).
      Lower k-fold accuracy reflects